In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
print(df.shape)
print(df["redirect_url"].iloc[0])

In [ ]:
from urllib.parse import urlparse

print(df["redirect_url"].apply(lambda u: urlparse(u).netloc).value_counts())

In [ ]:
import requests
import time
from urllib.parse import urlparse

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

sample = df.sample(10, random_state=42)

results = []
for i, row in sample.iterrows():
    try:
        r = requests.get(row["redirect_url"], headers=HEADERS, timeout=15)
        results.append({
            "id": row["id"],
            "title": row["title"],
            "status": r.status_code,
            "final_domain": urlparse(r.url).netloc,
            "length": len(r.text),
        })
        print(f"{r.status_code}  {urlparse(r.url).netloc:35s} {len(r.text):>8d}  {row['title'][:40]}")
    except Exception as e:
        results.append({"id": row["id"], "title": row["title"],
                        "status": None, "final_domain": str(e)[:60], "length": 0})
        print(f"ERR  {str(e)[:60]}")
    time.sleep(2)

In [ ]:
row = sample.iloc[1]
r = requests.get(row["redirect_url"], headers=HEADERS, timeout=15)
print(r.status_code, len(r.text))
print(row["title"])
print("-" * 60)

snippet = df[df["id"] == row["id"]]
print("CSV 里的 description 开头（用于比对）:")
print(row["title"][:60])

In [ ]:
print(row["redirect_url"])

In [ ]:
html = r.text.lower()
for kw in ["responsibilities", "requirements", "experience", "sponsorship", "visa"]:
    print(f"{kw:20s} {html.count(kw)}")

In [ ]:
%pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

idx = r.text.lower().find("responsibilities")
print(r.text[idx-1500:idx+500])

In [ ]:
for tag in soup.find_all(["section", "article", "div"], class_=True):
    cls = " ".join(tag.get("class"))
    if "desc" in cls.lower() or "adp" in cls.lower() or "job" in cls.lower():
        print(cls, "|", len(tag.get_text()))

In [ ]:
def extract_jd(html):
    """从 Adzuna 岗位页 HTML 里抽出 JD 正文，失败返回 None"""
    soup = BeautifulSoup(html, "html.parser")
    body = soup.find(class_="adp-body")
    if body is None:
        return None
    return body.get_text(separator=" ", strip=True)


text = extract_jd(r.text)
print(len(text))
print("-" * 60)
print(text[:600])

In [ ]:
for kw in ["responsibilities", "requirements", "python", "sponsorship", "visa"]:
    print(f"{kw:20s} {text.lower().count(kw)}")

In [ ]:
targets = df.sample(150, random_state=42).copy()
print(len(targets))
print(targets["region"].value_counts())

In [ ]:
import random

jd_texts = {}
failed = []

for n, (i, row) in enumerate(targets.iterrows(), 1):
    try:
        r = requests.get(row["redirect_url"], headers=HEADERS, timeout=20)
        if r.status_code == 200:
            t = extract_jd(r.text)
            if t:
                jd_texts[row["id"]] = t
            else:
                failed.append((row["id"], "no adp-body"))
        else:
            failed.append((row["id"], r.status_code))
    except Exception as e:
        failed.append((row["id"], str(e)[:40]))

    if n % 25 == 0:
        print(f"{n}/150  ok={len(jd_texts)}  failed={len(failed)}")

    time.sleep(random.uniform(2, 4))

print("-" * 40)
print("成功:", len(jd_texts), " 失败:", len(failed))

In [ ]:
from collections import Counter
print(Counter(reason for _, reason in failed))

In [ ]:
ok_ids = set(jd_texts.keys())
targets["fetched"] = targets["id"].isin(ok_ids)

print(targets.groupby("region")["fetched"].agg(["sum", "count", "mean"]))
print()
print(targets.groupby("search_keyword")["fetched"].agg(["sum", "count", "mean"]))

In [ ]:
jd_df = pd.DataFrame(
    [{"id": k, "jd_text": v} for k, v in jd_texts.items()]
)
jd_df.to_csv("../data/raw/jd_full_text_20260809.csv", index=False)
print(jd_df.shape)

In [ ]:
targets = targets.reset_index(drop=True)
targets["order"] = range(len(targets))
targets["half"] = targets["order"].apply(lambda x: "first75" if x < 75 else "last75")

print(pd.crosstab(targets["search_keyword"], targets["half"]))
print()
print(targets.groupby("half")["fetched"].mean())